[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kewel-corp/book-vol2/blob/main/notebooks/ch46_code.ipynb)

# 第46章　画像品質とアーチファクト：「品質に問題のある画像」をどう扱うか

**『医療診断支援AI開発　第2巻　応用編 ― 本格実装から臨床導入まで（第2巻　応用編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol2

## 品質スコアを、複数の指標から組み立てる

In [ ]:
def quality_flags(img, hdr, modality):
    """閾値はモダリティごとに分ける。CTとMRでは画素値の意味も分布もまったく違う。"""
    th = THRESHOLDS[modality]          # {"blur":..., "contrast":...} をモダリティ別に持つ
    flags = []
    # 1) ぼけ: ラプラシアン分散が低い=ぼけた/平坦な画像。ただしノイズが多いと逆に上がるので、
    #    「値が高い＝鮮明」とは限らない。低い値の画像だけを除外する用途に留める。
    if laplacian_var(img) < th["blur"]:
        flags.append("blurry")
    # 2) コントラスト不足: 画素値の分布の広がりが狭い。解剖や表示窓にも左右されるため、
    #    単一の閾値を「読影可否の保証」として扱わない。
    if img.std() < th["contrast"]:
        flags.append("low_contrast")
    # 3) 撮影条件の不一致: 造影相・スライス厚を確認。
    #    Enhanced CTなどでは、Contrast/Bolus Usage Sequence内の
    #    ContrastBolusAgentPhase(0018,9344)を利用できる場合がある。
    #    未記録なら造影剤情報・投与からの遅延時間・シリーズ記述等で推定し、
    #    判別できない場合は自動判定を保留する。
    phase = estimate_phase(hdr)                  # 判別できなければ None を返す前提
    if phase is None:
        flags.append("phase_unknown")           # 判別不能を「相違」と同じ扱いにしない（保留）
    elif phase != EXPECTED_PHASE:
        flags.append("wrong_phase")
    # 4) 対象範囲外: 想定臓器の存在を簡易な検出器で確認
    if not organ_present(img):
        flags.append("out_of_fov")
    return flags